In [1]:
""" Exercise 2 - Backpropagation Neural Network Prediction and Feature Analysis System
Dataset : Breast Cancer Wisconsin (Diagnostic)  -  569 samples, 30 features

Networks are written from scratch with NumPy (forward pass, sigmoid activation,
backpropagation of error, gradient-descent weight update).  scikit-learn is used
only to load the bundled copy of the dataset.

Data source priority
  1. a CSV path given on the command line
  2. data.csv / wdbc.data found next to this script
  3. sklearn.datasets.load_breast_cancer()  (offline, ships with sklearn)

Class encoding (as required):  Benign = 0, Malignant = 1
Note: sklearn's own encoding is the opposite, so it is remapped on load.
"""

import os
import sys
import time

import numpy as np

# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #
SEED = 42
TEST_RATIO = 0.30
THRESHOLD = 0.50
CLASS_NAMES = {0: "Benign", 1: "Malignant"}

# Confidence bands measured as distance of the sigmoid output from 0.5
LOW_BAND = 0.15        # |output - 0.5| <  0.15          -> Low
MOD_BAND = 0.35        # 0.15 <= |output - 0.5| < 0.35   -> Moderate
                       # |output - 0.5| >= 0.35          -> High

# Architectures (all three are trained with backpropagation; they differ in depth)
ARCH = {
    "BPNN":  {"hidden": [16, 8],     "lr": 0.30, "epochs": 400, "batch": 32},
    "MLP":   {"hidden": [32, 16, 8], "lr": 0.30, "epochs": 400, "batch": 32},
    "SHLNN": {"hidden": [8],         "lr": 0.30, "epochs": 400, "batch": 32},
}

CANDIDATE_FILES = [
    "data.csv", "wdbc.data", "breast_cancer.csv",
    "/mnt/user-data/uploads/data.csv",
    "/mnt/user-data/uploads/wdbc.data",
]

UCI_FEATURE_NAMES = [
    f"{stat} {base}" for stat in ["mean", "SE", "worst"]
    for base in ["radius", "texture", "perimeter", "area", "smoothness",
                 "compactness", "concavity", "concave points", "symmetry",
                 "fractal dim"]
]


# --------------------------------------------------------------------------- #
# 1. Data handling
# --------------------------------------------------------------------------- #
def _load_csv(path):
    """Kaggle data.csv (id,diagnosis,f1..f30) or UCI wdbc.data (id,M/B,f1..f30)."""
    rows, header = [], None
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",")
            if header is None and any(c.isalpha() for c in parts[0]):
                header = parts
                continue
            rows.append(parts)

    diag_col = 1
    labels = np.array([1 if r[diag_col].strip().upper().startswith("M") else 0
                       for r in rows], dtype=int)
    feats = []
    for r in rows:
        vals = []
        for v in r[diag_col + 1:]:
            v = v.strip()
            if v == "":
                continue
            try:
                vals.append(float(v))
            except ValueError:
                pass
        feats.append(vals[:30])
    X = np.array(feats, dtype=float)

    if header and len(header) >= 32:
        names = [h.strip() for h in header[diag_col + 1: diag_col + 31]]
    else:
        names = UCI_FEATURE_NAMES
    return X, labels, names, os.path.basename(path)


def load_dataset(path=None):
    """Return (X, y, feature_names, source).  Benign = 0, Malignant = 1."""
    paths = [path] if path else CANDIDATE_FILES
    for p in paths:
        if p and os.path.exists(p):
            return _load_csv(p)
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer()
    # sklearn: 0 = malignant, 1 = benign  ->  flip to match the required encoding
    y = 1 - d.target
    return d.data, y.astype(int), list(d.feature_names), "sklearn bundled copy"


def stratified_split(X, y, test_ratio=TEST_RATIO, seed=SEED):
    rng = np.random.default_rng(seed)
    tr, te = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        cut = int(round(len(idx) * (1 - test_ratio)))
        tr.extend(idx[:cut])
        te.extend(idx[cut:])
    tr, te = np.array(tr), np.array(te)
    rng.shuffle(tr)
    rng.shuffle(te)
    return X[tr], X[te], y[tr], y[te]


class MinMaxScaler:
    """Min-max normalisation fitted on the training set only."""

    def fit(self, X):
        self.lo = X.min(axis=0)
        self.hi = X.max(axis=0)
        self.span = np.where(self.hi - self.lo == 0, 1.0, self.hi - self.lo)
        return self

    def transform(self, X):
        return np.clip((X - self.lo) / self.span, 0.0, 1.0)


# --------------------------------------------------------------------------- #
# 2. Neural network (forward pass + backpropagation, written from scratch)
# --------------------------------------------------------------------------- #
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -60, 60)))


class NeuralNetwork:
    """Fully connected feed-forward net, sigmoid everywhere, trained by
    backpropagation of error with mini-batch gradient descent."""

    def __init__(self, hidden, lr=0.3, epochs=400, batch=32, seed=SEED, name="NN"):
        self.hidden = list(hidden)
        self.lr = lr
        self.epochs = epochs
        self.batch = batch
        self.seed = seed
        self.name = name
        self.W, self.b = [], []
        self.loss_history = []
        self.train_time = 0.0

    # ---- construction ---------------------------------------------------- #
    def _init_params(self, n_in):
        rng = np.random.default_rng(self.seed)
        sizes = [n_in] + self.hidden + [1]
        self.W, self.b = [], []
        for i in range(len(sizes) - 1):
            limit = np.sqrt(6.0 / (sizes[i] + sizes[i + 1]))   # Xavier
            self.W.append(rng.uniform(-limit, limit, (sizes[i], sizes[i + 1])))
            self.b.append(np.zeros((1, sizes[i + 1])))
        self.sizes = sizes

    # ---- forward pass ---------------------------------------------------- #
    def forward(self, X):
        """Return the list of layer activations, a[0] = input ... a[-1] = output."""
        a = [np.atleast_2d(X)]
        for W, b in zip(self.W, self.b):
            a.append(sigmoid(a[-1] @ W + b))
        return a

    def predict_proba(self, X):
        return self.forward(X)[-1].ravel()

    def predict(self, X, threshold=THRESHOLD):
        return (self.predict_proba(X) >= threshold).astype(int)

    def hidden_activations(self, X, layer=0):
        """Activations of the given hidden layer (0 = first hidden layer)."""
        return self.forward(X)[layer + 1]

    # ---- training -------------------------------------------------------- #
    def fit(self, X, y, verbose=False):
        start = time.time()
        X = np.atleast_2d(X)
        y = np.asarray(y, dtype=float).reshape(-1, 1)
        self._init_params(X.shape[1])
        rng = np.random.default_rng(self.seed)
        n = len(y)
        self.loss_history = []

        for epoch in range(1, self.epochs + 1):
            order = rng.permutation(n)
            for s in range(0, n, self.batch):
                idx = order[s:s + self.batch]
                self._step(X[idx], y[idx])
            out = self.predict_proba(X).reshape(-1, 1)
            loss = -np.mean(y * np.log(out + 1e-12) +
                            (1 - y) * np.log(1 - out + 1e-12))
            self.loss_history.append(float(loss))
            if verbose and (epoch % 50 == 0 or epoch == 1):
                print(f"    epoch {epoch:>4}  loss {loss:.6f}")
        self.train_time = time.time() - start
        return self

    def _step(self, Xb, yb):
        """One backpropagation update on a mini-batch."""
        a = self.forward(Xb)                       # forward pass
        m = len(yb)

        # output layer error  (BCE + sigmoid -> delta = a - y)
        delta = (a[-1] - yb) / m
        grads_W, grads_b = [None] * len(self.W), [None] * len(self.b)

        for l in range(len(self.W) - 1, -1, -1):   # propagate error backwards
            grads_W[l] = a[l].T @ delta
            grads_b[l] = delta.sum(axis=0, keepdims=True)
            if l > 0:
                # sigmoid derivative  a*(1-a)
                delta = (delta @ self.W[l].T) * a[l] * (1.0 - a[l])

        for l in range(len(self.W)):               # gradient-descent update
            self.W[l] -= self.lr * grads_W[l]
            self.b[l] -= self.lr * grads_b[l]

    def n_params(self):
        return sum(w.size for w in self.W) + sum(bb.size for bb in self.b)

    def describe(self):
        return " - ".join(str(s) for s in self.sizes)


# --------------------------------------------------------------------------- #
# 3. Metrics and presentation helpers
# --------------------------------------------------------------------------- #
def accuracy(y, p):
    return float(np.mean(y == p) * 100.0)


def confusion(y, p):
    tp = int(np.sum((p == 1) & (y == 1)))
    tn = int(np.sum((p == 0) & (y == 0)))
    fp = int(np.sum((p == 1) & (y == 0)))
    fn = int(np.sum((p == 0) & (y == 1)))
    return tn, fp, fn, tp


def metrics(y, p):
    tn, fp, fn, tp = confusion(y, p)
    prec = tp / (tp + fp) * 100 if tp + fp else 0.0
    rec = tp / (tp + fn) * 100 if tp + fn else 0.0
    spec = tn / (tn + fp) * 100 if tn + fp else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"acc": accuracy(y, p), "prec": prec, "rec": rec,
            "spec": spec, "f1": f1, "cm": (tn, fp, fn, tp)}


def confidence_category(output):
    d = abs(output - THRESHOLD)
    if d < LOW_BAND:
        return "Low"
    if d < MOD_BAND:
        return "Moderate"
    return "High"


def banner(title):
    print("\n" + "=" * 80)
    print(title.center(80))
    print("=" * 80)


def pause():
    input("\nPress <Enter> to return to the menu...")


# --------------------------------------------------------------------------- #
# 4. Model cache - each network is trained once and reused
# --------------------------------------------------------------------------- #
def get_model(D, name):
    if name in D["models"]:
        return D["models"][name]
    cfg = ARCH[name]
    print(f"\n[training {name}  layers {[D['Xtr'].shape[1]] + cfg['hidden'] + [1]}"
          f"  lr={cfg['lr']}  epochs={cfg['epochs']} ...]")
    net = NeuralNetwork(cfg["hidden"], cfg["lr"], cfg["epochs"], cfg["batch"],
                        name=name).fit(D["Xtr"], D["ytr"])
    print(f"[done in {net.train_time:.2f}s  final loss {net.loss_history[-1]:.6f}]")
    D["models"][name] = net
    return net


# --------------------------------------------------------------------------- #
# Option 1 - Tumor Classification using BPNN
# --------------------------------------------------------------------------- #
def option_1(D):
    banner("1. TUMOR CLASSIFICATION USING BPNN")
    net = get_model(D, "BPNN")
    out = net.predict_proba(D["Xte"])
    pred = (out >= THRESHOLD).astype(int)
    m = metrics(D["yte"], pred)

    print(f"\nArchitecture    : {net.describe()}  (sigmoid activation everywhere)")
    print(f"Trainable params: {net.n_params()}")
    print(f"Training        : backpropagation, mini-batch GD, "
          f"lr={net.lr}, {net.epochs} epochs")
    print(f"Samples         : train {len(D['ytr'])} / test {len(D['yte'])}")

    print("\n{:>6} {:>12} {:>12} {:>15} {:>10}".format(
        "S.No", "Actual", "Predicted", "Network Output", "Result"))
    print("-" * 80)
    show = min(25, len(out))
    for i in range(show):
        print("{:>6} {:>12} {:>12} {:>15.6f} {:>10}".format(
            i + 1, CLASS_NAMES[D["yte"][i]], CLASS_NAMES[pred[i]],
            out[i], "OK" if pred[i] == D["yte"][i] else "WRONG"))
    if len(out) > show:
        print(f"... {len(out) - show} more test samples not shown ...")

    tn, fp, fn, tp = m["cm"]
    print("-" * 80)
    print("Confusion matrix            Pred Benign   Pred Malignant")
    print(f"   Actual Benign            {tn:>11} {fp:>16}")
    print(f"   Actual Malignant         {fn:>11} {tp:>16}")
    print(f"\nMisclassified   : {fp + fn} / {len(out)}")
    print(f"Test accuracy   : {m['acc']:.2f} %")
    print(f"Train accuracy  : {accuracy(D['ytr'], net.predict(D['Xtr'])):.2f} %")
    print(f"Precision {m['prec']:.2f} %   Recall/Sensitivity {m['rec']:.2f} %   "
          f"Specificity {m['spec']:.2f} %   F1 {m['f1']:.2f} %")
    pause()


# --------------------------------------------------------------------------- #
# Option 2 - Prediction Confidence Analyzer
# --------------------------------------------------------------------------- #
def option_2(D):
    banner("2. PREDICTION CONFIDENCE ANALYZER  (MLP)")
    net = get_model(D, "MLP")
    out = net.predict_proba(D["Xte"])
    pred = (out >= THRESHOLD).astype(int)
    cats = [confidence_category(o) for o in out]

    print(f"\nModel          : MLP {net.describe()}")
    print(f"Threshold      : {THRESHOLD}")
    print(f"Confidence bands by |output - {THRESHOLD}|:")
    print(f"   Low      : < {LOW_BAND}          (output {0.5 - LOW_BAND:.2f} "
          f"- {0.5 + LOW_BAND:.2f})")
    print(f"   Moderate : {LOW_BAND} to {MOD_BAND}")
    print(f"   High     : >= {MOD_BAND}         (output < {0.5 - MOD_BAND:.2f} "
          f"or > {0.5 + MOD_BAND:.2f})")

    print("\n{:>6} {:>15} {:>12} {:>12} {:>12} {:>8}".format(
        "S.No", "Network Output", "Predicted", "Distance", "Confidence", "Actual"))
    print("-" * 80)
    show = min(25, len(out))
    for i in range(show):
        print("{:>6} {:>15.6f} {:>12} {:>12.4f} {:>12} {:>8}".format(
            i + 1, out[i], CLASS_NAMES[pred[i]], abs(out[i] - THRESHOLD),
            cats[i], CLASS_NAMES[D["yte"][i]]))
    if len(out) > show:
        print(f"... {len(out) - show} more test samples not shown ...")

    print("-" * 80)
    print("{:<12} {:>8} {:>10} {:>18}".format(
        "Category", "Count", "Share", "Accuracy in band"))
    for cat in ["Low", "Moderate", "High"]:
        idx = [i for i, c in enumerate(cats) if c == cat]
        if idx:
            acc = accuracy(D["yte"][idx], pred[idx])
            print("{:<12} {:>8} {:>9.1f}% {:>17.2f}%".format(
                cat, len(idx), len(idx) / len(out) * 100, acc))
        else:
            print("{:<12} {:>8} {:>9.1f}% {:>18}".format(cat, 0, 0.0, "-"))
    print("\nAccuracy normally rises with the confidence band; the Low band is where")
    print("the network is effectively undecided and should be flagged for review.")
    pause()


# --------------------------------------------------------------------------- #
# Option 3 - Feature Contribution Analyzer
# --------------------------------------------------------------------------- #
def option_3(D):
    banner("3. FEATURE CONTRIBUTION ANALYZER")
    net = get_model(D, "BPNN")
    n_test = len(D["yte"])
    raw = input(f"\nSelect a test sample number (1-{n_test}, blank = 1): ").strip()
    try:
        s = int(raw) - 1 if raw else 0
    except ValueError:
        print("Invalid input.")
        return pause()
    if not 0 <= s < n_test:
        print(f"Sample number must be between 1 and {n_test}.")
        return pause()

    x = D["Xte"][s].copy()
    base = float(net.predict_proba(x.reshape(1, -1))[0])
    print(f"\nSample #{s + 1}   actual = {CLASS_NAMES[D['yte'][s]]}   "
          f"baseline output = {base:.6f}   "
          f"predicted = {CLASS_NAMES[int(base >= THRESHOLD)]}")
    print("One feature at a time is swept over its normalised range [0,1] in 11")
    print("steps while every other feature is held at this sample's own value.")
    print("  Sweep range = max output - min output over the sweep (global effect)")
    print("  Local delta = |output(x+0.1) - output(x-0.1)| around the actual value")

    grid = np.linspace(0.0, 1.0, 11)
    rows = []
    for f in range(len(x)):
        Xs = np.tile(x, (len(grid), 1))
        Xs[:, f] = grid
        outs = net.predict_proba(Xs)

        Xl = np.tile(x, (2, 1))
        Xl[0, f] = np.clip(x[f] - 0.1, 0.0, 1.0)
        Xl[1, f] = np.clip(x[f] + 0.1, 0.0, 1.0)
        lo_hi = net.predict_proba(Xl)

        rows.append({
            "i": f, "name": D["names"][f],
            "omin": float(outs.min()), "omax": float(outs.max()),
            "rng": float(outs.max() - outs.min()),
            "local": float(abs(lo_hi[1] - lo_hi[0])),
            "signed": float(outs[-1] - outs[0]),
        })
    rows.sort(key=lambda r: (-r["rng"], -r["local"]))

    print("\n{:>4} {:<26} {:>9} {:>9} {:>12} {:>12} {:>10}".format(
        "Rank", "Feature", "min out", "max out", "Sweep Range", "Local Delta",
        "Direction"))
    print("-" * 80)
    for r, row in enumerate(rows, 1):
        arrow = "increases" if row["signed"] > 0 else "decreases"
        print("{:>4} {:<26} {:>9.5f} {:>9.5f} {:>12.5f} {:>12.5f} {:>10}".format(
            r, row["name"][:26], row["omin"], row["omax"], row["rng"],
            row["local"], arrow))

    print("-" * 80)
    print("TOP FIVE MOST INFLUENTIAL FEATURES FOR THIS SAMPLE")
    for r, row in enumerate(rows[:5], 1):
        direction = ("raising it pushes the output toward Malignant"
                     if row["signed"] > 0 else
                     "raising it pushes the output toward Benign")
        print(f"  {r}. {row['name']:<26} range {row['rng']:.5f}  "
              f"local {row['local']:.5f}  ({direction})")

    by_local = sorted(rows, key=lambda r: -r["local"])[:5]
    print("\nTOP FIVE BY LOCAL SENSITIVITY (smallest change needed to matter)")
    for r, row in enumerate(by_local, 1):
        print(f"  {r}. {row['name']:<26} local delta {row['local']:.5f}")

    if base < 0.05 or base > 0.95:
        print("\nThis sample sits deep in a saturated region of the sigmoid, so many")
        print("features show a small local delta - the prediction is robust here.")
    else:
        print("\nThis sample sits near the threshold, so several features can flip it;")
        print("sweep ranges cluster together and the local delta is the better guide.")
    print("\nNote: this is a local explanation - it describes this one sample,")
    print("not the global importance of the feature across the whole dataset.")
    pause()


# --------------------------------------------------------------------------- #
# Option 4 - Hidden Neuron Response Analyzer
# --------------------------------------------------------------------------- #
def option_4(D):
    banner("4. HIDDEN NEURON RESPONSE ANALYZER  (SHLNN)")
    net = get_model(D, "SHLNN")
    H = net.hidden_activations(D["Xte"], layer=0)
    n_neurons = H.shape[1]
    y = D["yte"]
    ben, mal = H[y == 0], H[y == 1]

    print(f"\nModel            : SHLNN {net.describe()}  (one hidden layer)")
    print(f"Hidden neurons   : {n_neurons}")
    print(f"Activation matrix: {H.shape[0]} test samples x {n_neurons} neurons")

    print("\nHidden neuron activation matrix (first 20 test samples)")
    head = "{:>6} {:>10}".format("S.No", "Class") + \
           "".join(f"{'N' + str(j + 1):>9}" for j in range(n_neurons))
    print(head)
    print("-" * len(head))
    for i in range(min(20, len(H))):
        row = "{:>6} {:>10}".format(i + 1, CLASS_NAMES[y[i]])
        row += "".join(f"{H[i, j]:>9.4f}" for j in range(n_neurons))
        print(row)
    if len(H) > 20:
        print(f"... {len(H) - 20} more rows not shown ...")

    print("\nAverage activation per neuron by class")
    print("{:>8} {:>14} {:>16} {:>14} {:>14}".format(
        "Neuron", "Avg (Benign)", "Avg (Malignant)", "Difference", "Prefers"))
    print("-" * 80)
    for j in range(n_neurons):
        b, mm = ben[:, j].mean(), mal[:, j].mean()
        pref = "Malignant" if mm > b else "Benign"
        print("{:>8} {:>14.5f} {:>16.5f} {:>14.5f} {:>14}".format(
            f"N{j + 1}", b, mm, mm - b, pref))

    jb, jm = int(np.argmax(ben.mean(axis=0))), int(np.argmax(mal.mean(axis=0)))
    jsep = int(np.argmax(np.abs(mal.mean(axis=0) - ben.mean(axis=0))))
    print("-" * 80)
    print(f"Highest average activation for BENIGN samples    : "
          f"N{jb + 1} ({ben[:, jb].mean():.5f})")
    print(f"Highest average activation for MALIGNANT samples : "
          f"N{jm + 1} ({mal[:, jm].mean():.5f})")
    print(f"Most class-discriminative neuron (largest gap)   : "
          f"N{jsep + 1} (gap {abs(mal[:, jsep].mean() - ben[:, jsep].mean()):.5f})")
    print(f"\nOutput-layer weight from each hidden neuron:")
    w_out = net.W[-1].ravel()
    print("  " + "   ".join(f"N{j + 1}={w_out[j]:+.4f}" for j in range(n_neurons)))
    print("A positive output weight means that neuron firing pushes the")
    print("prediction toward Malignant; a negative weight pushes toward Benign.")
    pause()


# --------------------------------------------------------------------------- #
# Option 5 - Input Feature Masking Challenge
# --------------------------------------------------------------------------- #
def masking_results(D, mode="mean"):
    net = get_model(D, "BPNN")
    base_out = net.predict_proba(D["Xte"])
    base_pred = (base_out >= THRESHOLD).astype(int)
    base_acc = accuracy(D["yte"], base_pred)
    rows = []
    for f in range(D["Xte"].shape[1]):
        Xm = D["Xte"].copy()
        Xm[:, f] = 0.0 if mode == "zero" else D["mean_norm"][f]
        out = net.predict_proba(Xm)
        pred = (out >= THRESHOLD).astype(int)
        rows.append({
            "i": f, "name": D["names"][f],
            "acc": accuracy(D["yte"], pred),
            "drop": base_acc - accuracy(D["yte"], pred),
            "delta": float(np.mean(np.abs(out - base_out))),
            "flips": int(np.sum(pred != base_pred)),
        })
    # accuracy is coarse on a small test set, so break ties with the mean
    # absolute change in the sigmoid output, which is continuous
    rows.sort(key=lambda r: (-r["drop"], -r["delta"]))
    return net, base_acc, rows


def option_5(D):
    banner("5. INPUT FEATURE MASKING CHALLENGE")
    mode = input("\nMask value - (1) dataset mean  (2) zero  [default 1]: ").strip()
    mode = "zero" if mode == "2" else "mean"
    net, base_acc, rows = masking_results(D, mode)

    print(f"\nModel           : BPNN {net.describe()} (weights unchanged, "
          f"no retraining)")
    print(f"Masking value   : {'0.0' if mode == 'zero' else 'training-set mean'}")
    print(f"Baseline accuracy with all 30 features: {base_acc:.2f} %")

    print("\n{:>4} {:<26} {:>10} {:>11} {:>14} {:>9}".format(
        "Rank", "Masked Feature", "Accuracy", "Drop (pts)", "Mean |d output|",
        "Flips"))
    print("-" * 80)
    for r, row in enumerate(rows, 1):
        print("{:>4} {:<26} {:>9.2f}% {:>11.2f} {:>14.5f} {:>9}".format(
            r, row["name"][:26], row["acc"], row["drop"], row["delta"],
            row["flips"]))

    top = rows[0]
    by_delta = sorted(rows, key=lambda r: -r["delta"])
    print("-" * 80)
    print(f"MAXIMUM PERFORMANCE DROP : {top['name']}  "
          f"({base_acc:.2f}% -> {top['acc']:.2f}%, "
          f"a loss of {top['drop']:.2f} points, {top['flips']} predictions flipped)")
    print(f"LARGEST OUTPUT DISTURBANCE: {by_delta[0]['name']}  "
          f"(mean |d output| = {by_delta[0]['delta']:.5f})")
    harmless = [r for r in rows if r["drop"] <= 0]
    print(f"Features whose masking does not hurt accuracy: {len(harmless)} of "
          f"{len(rows)}")
    print(f"\nAccuracy on {len(D['yte'])} test samples moves in steps of "
          f"{100 / len(D['yte']):.2f} points, so it is too coarse to separate")
    print("30 features. 'Mean |d output|' measures how far the sigmoid score moved")
    print("for the average sample and is used to break ties in the ranking above.")
    print("\nMany of the 30 features are strongly correlated (radius, perimeter and")
    print("area describe the same geometry), so masking one alone often changes")
    print("little - the network recovers the same information from its neighbours.")
    pause()


# --------------------------------------------------------------------------- #
# Option 6 - Misclassification Confidence Analyzer
# --------------------------------------------------------------------------- #
def option_6(D):
    banner("6. MISCLASSIFICATION CONFIDENCE ANALYZER")
    net = get_model(D, "BPNN")
    out = net.predict_proba(D["Xte"])
    pred = (out >= THRESHOLD).astype(int)
    wrong = np.where(pred != D["yte"])[0]

    print(f"\nModel            : BPNN {net.describe()}")
    print(f"Test samples     : {len(out)}")
    print(f"Misclassified    : {len(wrong)}  "
          f"(accuracy {accuracy(D['yte'], pred):.2f} %)")
    if len(wrong) == 0:
        print("\nNo misclassified samples on this test split.")
        return pause()

    print("\n{:>6} {:>12} {:>12} {:>15} {:>10} {:>12}".format(
        "S.No", "Actual", "Predicted", "Sigmoid Output", "Distance", "Confidence"))
    print("-" * 80)
    for i in wrong:
        print("{:>6} {:>12} {:>12} {:>15.6f} {:>10.4f} {:>12}".format(
            int(i) + 1, CLASS_NAMES[D["yte"][i]], CLASS_NAMES[pred[i]],
            out[i], abs(out[i] - THRESHOLD), confidence_category(out[i])))

    cats = [confidence_category(out[i]) for i in wrong]
    conf_wrong = [int(i) for i, c in zip(wrong, cats) if c == "High"]
    fn = [int(i) for i in wrong if D["yte"][i] == 1]
    fp = [int(i) for i in wrong if D["yte"][i] == 0]

    print("-" * 80)
    for cat in ["Low", "Moderate", "High"]:
        print(f"   {cat + ' confidence errors':<28}: {cats.count(cat)}")
    print(f"   {'False negatives (missed malignant)':<28}: {len(fn)} "
          f"-> samples {[i + 1 for i in fn]}")
    print(f"   {'False positives':<28}: {len(fp)} "
          f"-> samples {[i + 1 for i in fp]}")

    print("\nCONFIDENTLY WRONG ANALYSIS")
    print("-" * 80)
    if conf_wrong:
        print(f"{len(conf_wrong)} sample(s) were wrong while the output sat far from")
        print(f"the threshold (|output - 0.5| >= {MOD_BAND}): "
              f"{[i + 1 for i in conf_wrong]}")
        for i in conf_wrong:
            print(f"   sample {i + 1}: output {out[i]:.6f}, "
                  f"actual {CLASS_NAMES[D['yte'][i]]}, "
                  f"predicted {CLASS_NAMES[pred[i]]}")
        print("These are the dangerous errors - the sigmoid score gives no warning,")
        print("so a confidence filter would not catch them.")
    else:
        print("No confidently wrong predictions. Every error sits in the Low or")
        print("Moderate band near the threshold, which means a confidence filter")
        print("(for example, refer any sample with |output - 0.5| < 0.35 to a")
        print("human) would intercept all of them.")
    if fn:
        print("\nFalse negatives matter most clinically: a malignant tumour reported")
        print("as benign. Lowering the threshold below 0.5 trades specificity for")
        print("sensitivity and reduces them.")
    pause()


# --------------------------------------------------------------------------- #
# Option 7 - BPNN / MLP / SHLNN comparative analysis
# --------------------------------------------------------------------------- #
def option_7(D):
    banner("7. BPNN - MLP - SHLNN COMPARATIVE ANALYSIS")
    results = []
    for name in ["BPNN", "MLP", "SHLNN"]:
        net = get_model(D, name)
        out = net.predict_proba(D["Xte"])
        pred = (out >= THRESHOLD).astype(int)
        m = metrics(D["yte"], pred)
        cats = [confidence_category(o) for o in out]
        results.append({
            "name": name, "net": net, "out": out, "pred": pred, "m": m,
            "train_acc": accuracy(D["ytr"], net.predict(D["Xtr"])),
            "high": cats.count("High"), "low": cats.count("Low"),
            "wrong": int(np.sum(pred != D["yte"])),
        })

    print("\n--- A. Architecture and training cost ---")
    print("{:<8} {:<20} {:>9} {:>8} {:>8} {:>10} {:>12}".format(
        "Model", "Layers", "Params", "Epochs", "LR", "Time (s)", "Final Loss"))
    print("-" * 80)
    for r in results:
        n = r["net"]
        print("{:<8} {:<20} {:>9} {:>8} {:>8} {:>10.2f} {:>12.6f}".format(
            r["name"], n.describe(), n.n_params(), n.epochs, n.lr,
            n.train_time, n.loss_history[-1]))

    print("\n--- B. Classification performance on the test set ---")
    print("{:<8} {:>10} {:>10} {:>10} {:>10} {:>10} {:>8}".format(
        "Model", "Train Acc", "Test Acc", "Precision", "Recall", "F1", "Errors"))
    print("-" * 80)
    for r in results:
        m = r["m"]
        print("{:<8} {:>9.2f}% {:>9.2f}% {:>9.2f}% {:>9.2f}% {:>9.2f}% {:>8}".format(
            r["name"], r["train_acc"], m["acc"], m["prec"], m["rec"], m["f1"],
            r["wrong"]))

    print("\n--- C. Confusion matrices  (TN, FP, FN, TP) ---")
    print("{:<8} {:>10} {:>10} {:>10} {:>10} {:>22}".format(
        "Model", "TN", "FP", "FN", "TP", "False negatives matter"))
    print("-" * 80)
    for r in results:
        tn, fp, fn, tp = r["m"]["cm"]
        note = "missed malignant: " + str(fn)
        print("{:<8} {:>10} {:>10} {:>10} {:>10} {:>22}".format(
            r["name"], tn, fp, fn, tp, note))

    print("\n--- D. Output confidence profile ---")
    print("{:<8} {:>14} {:>16} {:>18}".format(
        "Model", "High conf.", "Low conf.", "Mean |out - 0.5|"))
    print("-" * 80)
    for r in results:
        print("{:<8} {:>14} {:>16} {:>18.4f}".format(
            r["name"], r["high"], r["low"],
            float(np.mean(np.abs(r["out"] - THRESHOLD)))))

    print("\n--- E. Convergence (training loss) ---")
    marks = [0, 49, 99, 199, 299, 399]
    marks = [m for m in marks if m < len(results[0]["net"].loss_history)]
    print("{:<8}".format("Model") + "".join(f"{'ep' + str(m + 1):>12}" for m in marks))
    print("-" * 80)
    for r in results:
        h = r["net"].loss_history
        print("{:<8}".format(r["name"]) + "".join(f"{h[m]:>12.6f}" for m in marks))

    print("\n--- F. Agreement between models ---")
    for i in range(3):
        for j in range(i + 1, 3):
            agree = float(np.mean(results[i]["pred"] == results[j]["pred"]) * 100)
            print(f"   {results[i]['name']:<6} vs {results[j]['name']:<6} : "
                  f"{agree:.2f} % identical predictions")

    best_acc = max(results, key=lambda r: r["m"]["acc"])
    best_rec = max(results, key=lambda r: r["m"]["rec"])
    fastest = min(results, key=lambda r: r["net"].train_time)
    smallest = min(results, key=lambda r: r["net"].n_params())

    print("\n--- G. Verdict ---")
    print(f"   Highest accuracy      : {best_acc['name']} "
          f"({best_acc['m']['acc']:.2f}%)")
    print(f"   Highest recall        : {best_rec['name']} "
          f"({best_rec['m']['rec']:.2f}%) - fewest missed malignant tumours")
    print(f"   Fastest to train      : {fastest['name']} "
          f"({fastest['net'].train_time:.2f}s)")
    print(f"   Smallest model        : {smallest['name']} "
          f"({smallest['net'].n_params()} parameters)")
    spread = (max(r["m"]["acc"] for r in results) -
              min(r["m"]["acc"] for r in results))
    print(f"   Accuracy spread       : {spread:.2f} points")
    tot_fn = sum(r["m"]["cm"][2] for r in results)
    print(f"""
   All three networks use the same backpropagation algorithm and the same
   sigmoid activation; only the depth differs (SHLNN one hidden layer, BPNN
   two, MLP three). The accuracy spread of {spread:.2f} points across a
   {len(D['yte'])}-sample test set is roughly
   {spread / (100 / len(D['yte'])):.0f} sample(s), so it is within noise:
   depth is not buying a meaningful gain here, while the MLP carries
   {results[1]['net'].n_params()} parameters against the SHLNN's
   {results[2]['net'].n_params()}. The models also agree on about 98% of
   predictions, which suggests they have all found essentially the same
   decision surface and that the dataset is close to linearly separable
   once normalised.

   Every model produces far more false negatives than false positives
   ({tot_fn} missed malignant cases in total). That is the wrong direction for
   a screening task, since a missed malignancy costs more than a false alarm.
   Lowering the decision threshold below {THRESHOLD} would trade specificity
   for sensitivity. Re-running with several random seeds is also worthwhile:
   the ranking above comes from one split and one initialisation.""")
    pause()


# --------------------------------------------------------------------------- #
# Menu driver
# --------------------------------------------------------------------------- #
MENU = """
+------------------------------------------------------------------+
|   BACKPROPAGATION NEURAL NETWORK PREDICTION & FEATURE ANALYSIS   |
|          Breast Cancer Wisconsin (Diagnostic) Dataset            |
+------------------------------------------------------------------+
|  1. Tumor Classification using BPNN                              |
|  2. Prediction Confidence Analyzer                               |
|  3. Feature Contribution Analyzer                                |
|  4. Hidden Neuron Response Analyzer                              |
|  5. Input Feature Masking Challenge                              |
|  6. Misclassification Confidence Analyzer                        |
|  7. BPNN-MLP-SHLNN Comparative Analysis                          |
|  8. Exit                                                         |
+------------------------------------------------------------------+"""


def build_data(path=None):
    X_raw, y, names, source = load_dataset(path)
    Xtr_raw, Xte_raw, ytr, yte = stratified_split(X_raw, y)
    scaler = MinMaxScaler().fit(Xtr_raw)
    Xtr, Xte = scaler.transform(Xtr_raw), scaler.transform(Xte_raw)
    return {
        "X_raw": X_raw, "y": y, "names": names, "source": source,
        "Xtr": Xtr, "Xte": Xte, "ytr": ytr, "yte": yte,
        "Xtr_raw": Xtr_raw, "Xte_raw": Xte_raw, "scaler": scaler,
        "mean_norm": Xtr.mean(axis=0), "models": {},
    }


def main():
    np.random.seed(SEED)
    path = sys.argv[1] if len(sys.argv) > 1 else None
    D = build_data(path)

    print(f"\nDataset source : {D['source']}")
    print(f"Shape          : {D['X_raw'].shape[0]} samples x "
          f"{D['X_raw'].shape[1]} features")
    print(f"Class encoding : Benign=0 ({int(np.sum(D['y'] == 0))} samples), "
          f"Malignant=1 ({int(np.sum(D['y'] == 1))} samples)")
    print(f"Split          : train {len(D['ytr'])} / test {len(D['yte'])} "
          f"(stratified, {int(TEST_RATIO * 100)}% test)")
    print(f"Preprocessing  : min-max normalisation to [0,1], fitted on train only")

    actions = {"1": option_1, "2": option_2, "3": option_3, "4": option_4,
               "5": option_5, "6": option_6, "7": option_7}
    while True:
        print(MENU)
        choice = input("Enter your choice (1-8): ").strip()
        if choice == "8":
            print("Exiting. Goodbye!")
            break
        if choice in actions:
            actions[choice](D)
        else:
            print("Invalid choice. Please enter a number between 1 and 8.")


if __name__ == "__main__":
    main()



Dataset source : sklearn bundled copy
Shape          : 569 samples x 30 features
Class encoding : Benign=0 (357 samples), Malignant=1 (212 samples)
Split          : train 398 / test 171 (stratified, 30% test)
Preprocessing  : min-max normalisation to [0,1], fitted on train only

+------------------------------------------------------------------+
|   BACKPROPAGATION NEURAL NETWORK PREDICTION & FEATURE ANALYSIS   |
|          Breast Cancer Wisconsin (Diagnostic) Dataset            |
+------------------------------------------------------------------+
|  1. Tumor Classification using BPNN                              |
|  2. Prediction Confidence Analyzer                               |
|  3. Feature Contribution Analyzer                                |
|  4. Hidden Neuron Response Analyzer                              |
|  5. Input Feature Masking Challenge                              |
|  6. Misclassification Confidence Analyzer                        |
|  7. BPNN-MLP-SHLNN Comparat